# Repeat Purchase Prediction, part 4
# Model benchmarking and optimisation

## Notebook purpose

This notebook compares more model families with the logistic regression and histogram gradient boosting from `03_modelling_homework.ipynb`, tunes them and picks the model for the final submission. It covers the modelling and evaluation phases of the project.

## Approach

- **Same validation as notebook 3:** four expanding yearly folds (2020–2023) with a 90-day gap, and the holdout from January 2024. Every model is scored on the same rows.
- **Defaults first:** each new model runs with its default settings and is compared with the untuned logistic regression and gradient boosting.
- **Then tuning:** each model is tuned on the development folds and compared with the tuned models of notebook 3.
- **One scoreboard:** F1 at each model's own threshold, the customer-bootstrap interval of the difference, folds where it is worse, ROC-AUC, balanced accuracy and log-loss.
- **Holdout only records:** choices are made on development results; the holdout scores the finalists and changes nothing.

## Structure

1. Setup and data
2. Reference models
3. New models with default settings
4. Tuning
5. Scoreboard and model choice
6. Threshold and calibration
7. Holdout assessment
8. Final fit and submission
9. Summary of decisions

# 1. Setup and data

## 1.1. Paths and experiment settings

In [33]:
# Handle paths without computer-specific locations.
from pathlib import Path

PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project. Open the notebook within the repository.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "preprocessed"
SUBMISSION_DIR = DATA_DIR / "submissions"
OPTUNA_DIR = DATA_DIR / "optuna"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

for folder in (SUBMISSION_DIR, OPTUNA_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# decision: a new experiment version, so 04 searches never mix with the homework ones
EXPERIMENT_VERSION = "project_v1"

N_TRIALS = 60         # completed trials per search
FRESH_SEARCH = False  # True deletes and restarts the named searches of this version

NEEDED = [
    PROCESSED_DIR / "train_clean.parquet",
    PROCESSED_DIR / "test_clean.parquet",
    PROCESSED_DIR / "decision_log_modelling.json",
    PROCESSED_DIR / "column_roles.json",
    PROCESSED_DIR / "preprocessing_config.json",
    RAW_DIR / "sample_submission.csv",
]

missing = [path.name for path in NEEDED if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}. Run 02_preprocessing and 03_modelling_homework first.")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Experiment version: {EXPERIMENT_VERSION} | trials per search: {N_TRIALS}")

Project root: ml-repeat-purchase
Experiment version: project_v1 | trials per search: 60


## 1.2. Imports, display and plot settings

In [34]:
# Standard library
import sys
import json
import hashlib
import warnings
from time import perf_counter

# Data and numerical calculations
import numpy as np
import pandas as pd

# Charts
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook display
from IPython.display import display

# Hyperparameter search
import optuna
from optuna.importance import FanovaImportanceEvaluator
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

# Pipelines and custom transformers
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.validation import check_is_fitted

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    f1_score,
    log_loss,
    roc_auc_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.exceptions import ConvergenceWarning

# Project modules
project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.repeat_purchase_prep as rpp
from src.decision_log import DecisionLog

RANDOM_STATE = rpp.RANDOM_STATE
target = rpp.TARGET

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [35]:
# Plot definitions shared by every chart in this notebook
ACCENT = {"burgundy": "#801333", "orange": "#ff8600", "near_black": "#1d212a"}
SERIES_COLOURS = ["#801333", "#f3a481", "#35193e", "#ff8600", "#e13342"]  # one colour per model or group
GRID_COLOUR = "#eeeeee"
FONT = "Inter, Arial, sans-serif"

# Seaborn's rocket palette as Plotly colour scales
ROCKET = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket", 11).as_hex())]
ROCKET_R = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket_r", 11).as_hex())]
# Correlations and loadings: always plotted with zmin=-1, zmax=1
DIVERGING = [[0.0, ACCENT["orange"]], [0.5, "#ffffff"], [1.0, ACCENT["burgundy"]]]

pio.templates["project"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=13, color=ACCENT["near_black"]),
        title=dict(x=0, xanchor="left", font=dict(size=17)),
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=SERIES_COLOURS,
        colorscale=dict(sequential=ROCKET_R, diverging=DIVERGING),
        xaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        yaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font_family=FONT),
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "project"

# decision: figures saved as HTML (plotly.js from the CDN) and PNG at 300 dpi equivalent
HTML_OPTIONS = dict(include_plotlyjs="cdn")
PNG_OPTIONS = dict(scale=300 / 72)

## 1.3. Data and folds

The cleaned data, column roles and validation settings come from notebook 2. The folds are the four expanding years used in notebook 3, so every model here is scored on the same rows. Only the key checks are repeated.

In [36]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")

config = json.loads((PROCESSED_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
roles = json.loads((PROCESSED_DIR / "column_roles.json").read_text(encoding="utf-8"))

# Validation settings saved by notebook 2
GAP_DAYS = int(config["validation"]["gap_days"])
HOLDOUT_START = pd.Timestamp(config["validation"]["holdout_start"])
DEV_END = HOLDOUT_START - pd.Timedelta(days=GAP_DAYS)
VAL_YEARS = list(config["validation"]["validation_years"])

dev = train.loc[train["purchase_date"] < DEV_END].reset_index(drop=True)
holdout = train.loc[train["purchase_date"] >= HOLDOUT_START].reset_index(drop=True)

# Expanding yearly folds: training outcomes close before the validation year starts
folds = []
for year in VAL_YEARS:
    start = pd.Timestamp(year=year, month=1, day=1)
    stop = min(pd.Timestamp(year=year + 1, month=1, day=1), DEV_END)
    train_idx = np.flatnonzero(dev["purchase_date"] < start - pd.Timedelta(days=GAP_DAYS))
    val_idx = np.flatnonzero((dev["purchase_date"] >= start) & (dev["purchase_date"] < stop))
    folds.append((train_idx, val_idx))

# Key checks only; the full set ran in notebook 3
assert set(train.columns) - {target} == set(test.columns)
assert set(sample_submission["ID"]) == set(test["ID"])
assert roles == rpp.column_roles(dev), "Saved column roles no longer match the data."

log = DecisionLog.load(PROCESSED_DIR / "decision_log_modelling.json")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Development: {len(dev):,} | Holdout: {len(holdout):,}")
print("Fold sizes (train / validation):", {y: (len(tr), len(va)) for y, (tr, va) in zip(VAL_YEARS, folds)})
print(f"Decisions loaded: {len(log)}")

Train: (6469, 53) | Test: (986, 52)
Development: 5,403 | Holdout: 835
Fold sizes (train / validation): {2020: (2578, 670), 2021: (3234, 782), 2022: (4004, 628), 2023: (4705, 536)}
Decisions loaded: 38


## 1.4. Pipeline builders

Two input branches, as decided in notebook 2. The linear branch fills, scales and encodes the columns, and can add the elastic-net selection. The tree branch keeps missing values and codes the categories. Distance and probability models (KNN, Naive Bayes, LDA) use the linear branch; trees use the tree branch.

In [37]:
# Saved branch decisions from notebook 2
NOISE = list(config["excluded_noise"])
TREE_DROP = list(config["hist_gradient_boosting"]["drop_columns"])


def branch_roles(branch):
    """Numeric, large and small categorical inputs of one branch."""
    excluded = set(NOISE + (TREE_DROP if branch == "tree" else []))
    return {group: [c for c in roles[group] if c not in excluded] for group in ("num", "high", "low")}


def branch_inputs(branch):
    r = branch_roles(branch)
    return r["num"] + r["high"] + r["low"]


def build_linear_pipeline(model, select=True):
    """Median fill and scaling, frequency and one-hot encoding, optional elastic-net selection."""
    settings, r = config["linear"], branch_roles("linear")
    prep = ColumnTransformer(
        [
            ("num", Pipeline([
                ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scale", StandardScaler()),
            ]), r["num"]),
            ("high", Pipeline([("encode", rpp.FrequencyEncoder()), ("scale", StandardScaler())]), r["high"]),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=settings["rare_min_rows"]), r["low"]),
        ],
        sparse_threshold=0,
    )
    steps = [("prep", prep)]
    if select:
        steps.append(("select", SelectFromModel(
            LogisticRegression(solver="saga", l1_ratio=settings["selector_l1_ratio"], C=settings["selector_C"],
                               max_iter=5000, random_state=RANDOM_STATE),
            threshold=settings["selector_threshold"],
        )))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def build_tree_pipeline(model):
    """Small categories as codes (first), large ones as frequencies, numbers untouched; missing values kept."""
    r = branch_roles("tree")
    prep = ColumnTransformer(
        [
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), r["low"]),
            ("high", rpp.FrequencyEncoder(), r["high"]),
            ("num", "passthrough", r["num"]),
        ],
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )
    model = clone(model)
    if isinstance(model, HistGradientBoostingClassifier):
        # decision: native categories and no internal random split, as in notebook 3
        model.set_params(categorical_features=list(range(len(r["low"]))), early_stopping=False)
    return Pipeline([("prep", prep), ("model", model)])


print("Linear inputs:", len(branch_inputs("linear")), "| Tree inputs:", len(branch_inputs("tree")))

Linear inputs: 41 | Tree inputs: 38


## 1.5. Scoring on the folds

Each model is fitted in every fold and scored on the later year. Its threshold is chosen once, on the four years' predictions pooled, with the same rule as notebook 3. The predictions keep the customer and whether that customer appeared in the training part, for the error analysis.

In [38]:
def best_threshold(y, p, tol=0.005):
    """Centre of the longest run of thresholds within tol of the best F1 (grid 0.05 to 0.95)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    actual = np.asarray(y).astype(bool)[:, None]
    predicted = np.asarray(p)[:, None] >= grid
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    f1s = 2 * tp / np.maximum(2 * tp + fp + fn, 1)

    near = np.flatnonzero(f1s >= f1s.max() - tol)
    runs = [r for r in np.split(near, np.flatnonzero(np.diff(near) > 1) + 1) if np.any(f1s[r] == f1s.max())]
    chosen = max(runs, key=lambda r: (len(r), -int(r[0])))
    return float(grid[chosen[(len(chosen) - 1) // 2]]), grid, f1s


def f1_floor(y):
    """F1 of predicting a repeat for every row."""
    share = np.mean(y)
    return 2 * share / (1 + share)


def score_folds(pipe, inputs, data=dev):
    """Fit in every fold; return (fold table, out-of-fold predictions, pooled threshold)."""
    rows, oof = [], []
    for (tr, va), year in zip(folds, VAL_YEARS):
        train_part, val_part = data.iloc[tr], data.iloc[va]
        start = perf_counter()
        model = clone(pipe).fit(train_part[inputs], train_part[target])
        seconds = perf_counter() - start

        p = model.predict_proba(val_part[inputs])[:, 1]
        p_train = model.predict_proba(train_part[inputs])[:, 1]
        y = val_part[target].to_numpy()
        steps = model.named_steps
        rows.append({
            "fold": year,
            "n_features": (int(steps["select"].get_support().sum()) if "select" in steps
                           else len(steps["prep"].get_feature_names_out())),
            "log_loss": log_loss(y, p, labels=[0, 1]),
            "train_log_loss": log_loss(train_part[target], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(y, p),
            "fit_seconds": seconds,
        })
        # Kept for later error analysis: who the customer is and whether training saw them
        oof.append(pd.DataFrame({
            "fold": year, "ID": val_part["ID"].to_numpy(), "customer_id": val_part["customer_id"].to_numpy(),
            "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(), "y": y, "p": p,
        }))

    oof = pd.concat(oof, ignore_index=True)
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold


def summary_row(table, oof, threshold):
    """One line per model: pooled F1 at its own threshold plus fold averages."""
    predicted = oof["p"] >= threshold
    return {
        "n_features": table["n_features"].mean(),
        "f1": f1_score(oof["y"], predicted),
        "floor": f1_floor(oof["y"]),
        "balanced_accuracy": balanced_accuracy_score(oof["y"], predicted),
        "roc_auc": table["roc_auc"].mean(),
        "log_loss": table["log_loss"].mean(),
        "overfit_gap": (table["log_loss"] - table["train_log_loss"]).mean(),
        "threshold": threshold,
        "fit_seconds": table["fit_seconds"].sum(),
    }

## 1.6. Decision rule

Same rule as notebook 3: an option beats the reference only if the 95% customer-bootstrap interval of the F1 difference sits above zero and it is worse in at most one fold.

In [39]:
def fast_f1(y, predicted):
    tp = np.sum(y & predicted)
    return 2 * tp / (2 * tp + np.sum(~y & predicted) + np.sum(y & ~predicted))


def bootstrap_f1_diff(oof_ref, oof_opt, thr_ref, thr_opt, n_boot=2000, seed=RANDOM_STATE):
    """95% interval of F1(option) - F1(reference), resampling whole customers."""
    assert oof_ref["ID"].equals(oof_opt["ID"]), "Options must be scored on the same rows."
    y = oof_ref["y"].to_numpy(dtype=bool)
    pred_ref, pred_opt = oof_ref["p"].to_numpy() >= thr_ref, oof_opt["p"].to_numpy() >= thr_opt
    groups = list(oof_ref.groupby("customer_id", sort=False).indices.values())

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        diffs[b] = fast_f1(y[rows], pred_opt[rows]) - fast_f1(y[rows], pred_ref[rows])
    return np.percentile(diffs, [2.5, 97.5])


def judge(runs, reference):
    """Scoreboard against the reference: better only if the interval is above zero and at most one fold is worse."""
    ref_table, ref_oof, ref_thr = runs[reference]
    rows = []
    for name, (table, oof, thr) in runs.items():
        row = {"model": name, **summary_row(table, oof, thr)}
        row["folds_worse"] = int((table["f1"] < ref_table["f1"]).sum())
        if name == reference:
            row.update(ci_low=np.nan, ci_high=np.nan, verdict="reference")
        else:
            lo, hi = bootstrap_f1_diff(ref_oof, oof, ref_thr, thr)
            verdict = ("better" if lo > 0 and row["folds_worse"] <= 1
                       else "worse" if hi < 0 else "no clear difference")
            row.update(ci_low=lo, ci_high=hi, verdict=verdict)
        rows.append(row)

    board = pd.DataFrame(rows).set_index("model")
    board.insert(2, "f1_diff", board["f1"] - board.loc[reference, "f1"])
    return board

# 2. Reference models

Three models set the bar for everything that follows. The dummy always predicts the training share of repeats, so its F1 is the "always repeat" floor. The untuned logistic regression and gradient boosting are the starting points of notebook 3. Logistic regression is the reference: a new model has to beat it under the decision rule.

## 2.1. Scoreboard

In [40]:
linear_inputs, tree_inputs = branch_inputs("linear"), branch_inputs("tree")

reference_pipes = {
    # decision: the dummy predicts the training share, so its F1 is the always-repeat floor
    "dummy": (build_linear_pipeline(DummyClassifier(strategy="prior"), select=False), linear_inputs),
    "logistic": (build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)), linear_inputs),
    "hist_gb": (build_tree_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)), tree_inputs),
}

reference_runs = {name: score_folds(pipe, inputs) for name, (pipe, inputs) in reference_pipes.items()}

board_reference = judge(reference_runs, reference="logistic")
board_reference.loc["dummy", "n_features"] = 0  # it ignores every column
display(board_reference)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
dummy,0.000,0.709,-0.042,0.709,0.500,0.500,0.689,0.002,0.300,0.173,4,-0.059,-0.021,worse
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
hist_gb,38.000,0.736,-0.015,0.709,0.654,0.760,0.607,0.348,0.300,1.227,4,-0.031,0.001,no clear difference


Logistic regression reaches F1 0.750 against the floor of 0.709, with ROC-AUC 0.784 and balanced accuracy 0.684. Gradient boosting scores 0.736: its interval [-0.031, 0.001] includes zero, so the difference is not clear, but it is worse in all four years. The dummy is clearly worse, as expected, with balanced accuracy 0.500 and ROC-AUC 0.500.

## 2.2. F1 by validation year

In [41]:
fold_f1 = pd.DataFrame({name: reference_runs[name][0]["f1"] for name in ["logistic", "hist_gb"]})
fold_f1["always repeat"] = [f1_floor(part["y"]) for _, part in reference_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in ["logistic", "hist_gb", "always repeat"]:
    fig.add_trace(go.Scatter(
        x=fold_f1.index, y=fold_f1[name], mode="lines+markers" if name != "always repeat" else "lines",
        name=name, line=dict(dash="dash", color="grey") if name == "always repeat" else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Reference models: F1 in each validation year",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis=dict(title="F1 at each model's pooled threshold"),
    height=420,
)
fig.write_html(FIGURES_DIR / "04_reference_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_reference_f1_by_year.png", **PNG_OPTIONS)
fig.show()

In [42]:
display(fold_f1.round(3))

,logistic,hist_gb,always repeat
fold,,,
2020,0.758,0.742,0.745
2021,0.764,0.764,0.709
2022,0.716,0.689,0.649
2023,0.760,0.740,0.728


Logistic regression stays above the floor in every year, from 0.716 in 2022 to 0.764 in 2021. Its smallest lift is in 2020 (0.758 against 0.745). Gradient boosting ties it in 2021 (0.764) and falls to 0.689 in 2022, the year with the lowest floor (0.649).

## Findings

| Model | F1 | ROC-AUC | Balanced accuracy | Verdict against logistic |
|---|---|---|---|---|
| Dummy (always repeat) | 0.709 | 0.500 | 0.500 | worse |
| Logistic regression | 0.750 | 0.784 | 0.684 | reference |
| Gradient boosting | 0.736 | 0.760 | 0.654 | no clear difference, worse in 4 of 4 years |

- The untuned logistic regression is the reference for the default models in section 3, at F1 0.750.
- The floor is high (0.709), because about 56% of development purchases are followed by a repeat. Balanced accuracy and ROC-AUC show the gap to a model without information more clearly than F1.
- Gradient boosting's log-loss gap between training and validation (0.348, against 0.009 for logistic regression) shows its default settings follow the training years closely.

# 3. New models with default settings

Each model runs with its library defaults on the same folds, so the comparison shows what each model family brings before any tuning. A short check before each model tests the assumption it relies on.

## 3.1. Naive Bayes

Gaussian Naive Bayes assumes that, within each class, every column is normal and independent of the others. The check measures how far the prepared columns are from that.

In [43]:
# Prepared linear-branch columns of the last fold, the same matrix Naive Bayes sees
tr, _ = folds[-1]
nb_prep = build_linear_pipeline(GaussianNB(), select=False).named_steps["prep"]
X_nb = pd.DataFrame(
    nb_prep.fit_transform(dev.iloc[tr][linear_inputs]),
    columns=nb_prep.get_feature_names_out(),
)
y_nb = dev.iloc[tr][target].to_numpy()

binary = [c for c in X_nb if set(np.unique(X_nb[c])) <= {0.0, 1.0}]
numeric = [c for c in X_nb if c not in binary]
skew_by_class = pd.DataFrame({
    "skew_no_repeat": X_nb.loc[y_nb == 0, numeric].skew(),
    "skew_repeat": X_nb.loc[y_nb == 1, numeric].skew(),
})
skew_by_class["max_abs_skew"] = skew_by_class.abs().max(axis=1)

# Strongest correlation of each numeric column with another, inside the repeat class
corr = X_nb.loc[y_nb == 1, numeric].corr().abs().where(lambda m: ~np.eye(len(m), dtype=bool))
skew_by_class["max_abs_corr"] = corr.max()

print(f"{len(X_nb.columns)} prepared columns: {len(binary)} take only 0 and 1, {len(numeric)} are numeric")
print(f"Numeric columns with |skew| > 1 in either class: {(skew_by_class['max_abs_skew'] > 1).sum()}")
print(f"Numeric columns correlated above 0.8 with another: {(skew_by_class['max_abs_corr'] > 0.8).sum()}")
display(skew_by_class.sort_values("max_abs_skew", ascending=False).head(8).round(2))

73 prepared columns: 39 take only 0 and 1, 34 are numeric
Numeric columns with |skew| > 1 in either class: 30
Numeric columns correlated above 0.8 with another: 17


,skew_no_repeat,skew_repeat,max_abs_skew,max_abs_corr
num__Total Desc_ Global,31.360,24.280,31.360,0.130
num__Total Portes,15.700,24.020,24.020,0.270
num__prior_std_gap_days,5.080,9.020,9.020,0.640
num__Total Desc_ Linha,7.840,6.460,7.840,0.630
num__invoice_count_current_purchase,7.610,5.690,7.610,0.170
num__spend_30d,7.100,3.820,7.100,0.710
num__prior_mean_gap_days,4.530,6.970,6.970,0.640
num__prior_purchase_count,6.330,2.980,6.330,0.870


Both assumptions fail. Of the 73 prepared columns, 39 take only the values 0 and 1, so they cannot be normal. Of the 34 numeric columns, 30 have a skew above 1 in at least one class (up to 31.4 for `Total Desc_ Global`), and 17 correlate above 0.8 with another column, so the same information is counted several times.

In [44]:
nb_runs = {
    "naive_bayes": score_folds(build_linear_pipeline(GaussianNB(), select=False), linear_inputs),
    "naive_bayes_selected": score_folds(build_linear_pipeline(GaussianNB()), linear_inputs),
}
board_nb = judge({"logistic": reference_runs["logistic"], **nb_runs}, reference="logistic")
display(board_nb)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
naive_bayes,69.250,0.673,-0.078,0.709,0.650,0.725,6.543,2.308,0.120,0.187,4,-0.124,-0.047,worse
naive_bayes_selected,20.000,0.703,-0.048,0.709,0.703,0.756,2.568,0.447,0.060,2.893,4,-0.098,-0.019,worse


On all 69 prepared columns Naive Bayes scores F1 0.673, below the always-repeat floor of 0.709. With the 20 elastic-net columns it reaches 0.703, still below the floor. Both are clearly worse than logistic regression. The probabilities are badly overconfident: log-loss is 6.54 and 2.57 against 0.561, and the selected thresholds sit at 0.12 and 0.06, near the bottom of the grid.

## 3.2. Linear discriminant analysis

LDA fits one normal distribution per class with a shared covariance matrix, and it has to invert that matrix. Strongly or exactly collinear columns make the inversion unstable, so the check measures collinearity.

In [45]:
# Collinearity of the same prepared matrix (last fold), all columns together
X_all = X_nb.to_numpy()
X_all = X_all[:, X_all.std(axis=0) > 0]  # constant columns carry no information
singular_values = np.linalg.svd(X_all - X_all.mean(axis=0), compute_uv=False)
condition_number = singular_values[0] / singular_values[-1]

# Pairs of numeric columns that move almost together
pairs = (
    X_nb[numeric].corr().abs()
    .where(np.triu(np.ones((len(numeric), len(numeric)), dtype=bool), k=1))
    .stack().sort_values(ascending=False)
)

print(f"Condition number of the centred matrix: {condition_number:,.0f}")
print(f"Numeric pairs with |correlation| > 0.9: {(pairs > 0.9).sum()}")
display(pairs.head(6).round(3).rename("abs_correlation").to_frame())

Condition number of the centred matrix: 91,724,876,045,469,680
Numeric pairs with |correlation| > 0.9: 5


,,abs_correlation
num__Total a Pagar,num__Total Liquido,0.996
num__Total Bruto,num__Total Liquido,0.982
num__Total a Pagar,num__Total Bruto,0.972
num__purchase_count_180d,num__purchase_count_365d,0.954
num__purchase_count_90d,num__purchase_count_180d,0.936
num__Total a Pagar,num__Total Imp_,0.894


The condition number is about 4 × 10¹⁷, which means the matrix is singular: some columns are exact combinations of others. Two causes: the one-hot encoding keeps every level, so each group of indicators sums to 1, and the accounting identity from the EDA makes `Total a Pagar` equal to `Total Liquido` plus `Total Imp_` (correlation 0.996). Five numeric pairs correlate above 0.9, including the 90-, 180- and 365-day purchase counts.

In [46]:
lda_runs = {
    "lda": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False), linear_inputs),
    "lda_selected": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis()), linear_inputs),
}
board_lda = judge({"logistic": reference_runs["logistic"], **lda_runs}, reference="logistic")
display(board_lda)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
lda,69.250,0.733,-0.018,0.709,0.645,0.741,0.616,0.052,0.350,0.296,3,-0.031,-0.006,worse
lda_selected,20.000,0.733,-0.017,0.709,0.613,0.752,0.593,0.021,0.290,2.823,4,-0.030,-0.002,worse


LDA scores F1 0.733 on all columns and 0.733 on the 20 selected ones, above the floor of 0.709 but clearly below logistic regression (intervals [-0.031, -0.006] and [-0.030, -0.002]). Its probabilities are far better behaved than Naive Bayes's: log-loss 0.616 and 0.593, with thresholds of 0.35 and 0.29. The default `svd` solver runs despite the singular matrix, because it never inverts the covariance directly.

## 3.3. K-nearest neighbours

KNN predicts from the training rows closest to each new row, so it relies on distances being meaningful. With many columns, the nearest and farthest neighbours can end up almost equally far away. The check compares the two distances on all prepared columns and on the elastic-net selection.

In [47]:
# Distance concentration on the last fold: as columns grow, nearest and farthest neighbours look alike
tr, va = folds[-1]
selected_pipe = build_linear_pipeline(KNeighborsClassifier()).fit(dev.iloc[tr][linear_inputs], dev.iloc[tr][target])

matrices = {
    "all columns": (X_nb.to_numpy(), nb_prep.transform(dev.iloc[va][linear_inputs])),
    "selected columns": tuple(
        selected_pipe.named_steps["select"].transform(selected_pipe.named_steps["prep"].transform(dev.iloc[part][linear_inputs]))
        for part in (tr, va)
    ),
}

rows = []
for name, (X_train_fold, X_val_fold) in matrices.items():
    distances = np.linalg.norm(X_val_fold[:, None, :] - X_train_fold[None, :, :], axis=2)
    nearest, farthest = distances.min(axis=1), distances.max(axis=1)
    rows.append({
        "columns": X_train_fold.shape[1],
        "median_nearest": np.median(nearest),
        "median_farthest": np.median(farthest),
        "median_contrast": np.median((farthest - nearest) / nearest),
    })
display(pd.DataFrame(rows, index=list(matrices)).round(2))

,columns,median_nearest,median_farthest,median_contrast
all columns,73,2.230,48.690,20.770
selected columns,22,1.120,17.230,14.500


The contrast between the farthest and nearest neighbour is higher with all 73 columns (20.8) than with the 22 selected ones (14.5), the opposite of the expected concentration. The farthest distances are driven by the extreme values seen in the Naive Bayes check (skew up to 31), so this measure mostly reflects outliers rather than the number of columns. The selection halves the distance to the nearest neighbour (median 1.12 against 2.23).

In [48]:
knn_runs = {
    "knn": score_folds(build_linear_pipeline(KNeighborsClassifier(), select=False), linear_inputs),
    "knn_selected": score_folds(build_linear_pipeline(KNeighborsClassifier()), linear_inputs),
}
board_knn = judge({"logistic": reference_runs["logistic"], **knn_runs}, reference="logistic")
display(board_knn)

print("Distinct probabilities from the selected model:", np.sort(knn_runs["knn_selected"][1]["p"].unique()))

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
knn,69.250,0.726,-0.024,0.709,0.632,0.734,1.981,1.557,0.300,0.277,4,-0.041,-0.008,worse
knn_selected,20.000,0.732,-0.018,0.709,0.643,0.731,2.130,1.709,0.300,3.195,3,-0.036,-0.001,worse


Distinct probabilities from the selected model: [0.  0.2 0.4 0.6 0.8 1. ]


With five neighbours, KNN scores F1 0.726 on all columns and 0.732 on the selected ones, both above the floor of 0.709 and clearly below logistic regression. It gives only six probabilities (0, 0.2, … 1), so its threshold of 0.30 means "at least two of the five neighbours repeated". Many predictions at exactly 0 or 1 explain the high log-loss (about 2.0), and the training-validation gap of 1.6–1.7 shows each training row votes for itself.

## 3.4. Decision tree

A tree with default settings keeps splitting until every leaf is pure, so it can memorise the training rows. The check measures how large the default tree grows in each fold and how many leaves hold a single row. Missing values stay in: the tree sends them down whichever branch fits the training rows better.

In [49]:
# Size of the default tree in each fold, fitted on the tree branch with missing values kept
tree_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    fitted = build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)).fit(
        dev.iloc[tr][tree_inputs], dev.iloc[tr][target]
    )
    tree = fitted.named_steps["model"]
    leaf_sizes = tree.tree_.n_node_samples[tree.tree_.children_left == -1]
    tree_rows.append({
        "fold": year,
        "training_rows": len(tr),
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "median_leaf_size": np.median(leaf_sizes),
        "single_row_leaves": np.mean(leaf_sizes == 1),
    })
display(pd.DataFrame(tree_rows).set_index("fold").round(3))

,training_rows,depth,leaves,median_leaf_size,single_row_leaves
fold,,,,,
2020,2578,24,463,3.000,0.348
2021,3234,26,583,2.000,0.341
2022,4004,26,693,2.000,0.375
2023,4705,33,830,2.000,0.402


The default tree grows with the data: from depth 24 and 463 leaves in the 2020 fold to depth 33 and 830 leaves in 2023. The median leaf holds two or three training rows, and 34–40% of the leaves hold a single row, so most answers rest on one or two purchases.

In [50]:
tree_runs = {"decision_tree": score_folds(build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), tree_inputs)}
board_tree = judge({"logistic": reference_runs["logistic"], **tree_runs}, reference="logistic")
display(board_tree)

print("Distinct probabilities:", tree_runs["decision_tree"][1]["p"].nunique())

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
decision_tree,38.000,0.626,-0.124,0.709,0.594,0.592,14.692,14.692,0.500,0.235,4,-0.161,-0.092,worse


Distinct probabilities: 2


The default tree scores F1 0.626, well below the floor of 0.709, with ROC-AUC 0.592 and balanced accuracy 0.594. Its pure leaves give only two probabilities, 0 and 1, so the threshold has no effect and log-loss is extreme (14.7). The training-validation gap equals the validation log-loss, because the tree fits its training rows perfectly.

## 3.5. Random forest

A random forest averages many deep trees, each grown on a bootstrap sample of the rows. The rows left out of each sample give an out-of-bag score at no extra cost, but those rows are drawn at random from the same years. The check compares that score with F1 on the later validation year, to see whether random-row scoring would flatter the model here.

In [51]:
# Out-of-bag F1 (random rows of the training years) against F1 on the later validation year
oob_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    forest_pipe = build_tree_pipeline(RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE, n_jobs=-1))
    forest_pipe.fit(dev.iloc[tr][tree_inputs], dev.iloc[tr][target])
    forest = forest_pipe.named_steps["model"]

    p_oob = forest.oob_decision_function_[:, 1]
    p_val = forest_pipe.predict_proba(dev.iloc[va][tree_inputs])[:, 1]
    oob_rows.append({
        "fold": year,
        "oob_f1": f1_score(dev.iloc[tr][target], p_oob >= 0.5),
        "validation_f1": f1_score(dev.iloc[va][target], p_val >= 0.5),
        "median_tree_depth": np.median([t.get_depth() for t in forest.estimators_]),
    })
oob_table = pd.DataFrame(oob_rows).set_index("fold")
oob_table["optimism"] = oob_table["oob_f1"] - oob_table["validation_f1"]
display(oob_table.round(3))

,oob_f1,validation_f1,median_tree_depth,optimism
fold,,,,
2020,0.723,0.722,25.000,0.001
2021,0.722,0.741,26.000,-0.018
2022,0.734,0.724,27.000,0.011
2023,0.728,0.703,28.000,0.025


At a 0.5 cut-off, the out-of-bag F1 differs from the later year's F1 by -0.018 to +0.025, with no consistent direction. On this data, random-row scoring does not clearly flatter the forest, but the chronological folds remain the basis for every decision. The trees grow to a median depth of 25–28, as deep as the single default tree.

In [52]:
forest_runs = {
    "random_forest": score_folds(
        build_tree_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)), tree_inputs
    )
}
board_forest = judge({"logistic": reference_runs["logistic"], **forest_runs}, reference="logistic")
display(board_forest)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,2.878,0,NaN,NaN,reference
random_forest,38.000,0.736,-0.015,0.709,0.669,0.760,0.589,0.433,0.440,0.778,4,-0.029,-0.002,worse


The default forest scores F1 0.736, the same as gradient boosting and far above the single tree's 0.626: averaging 100 deep trees removes most of their individual noise. It is still clearly worse than logistic regression (interval [-0.029, -0.002]) and below it in all four years. Its probabilities are usable (log-loss 0.589, 94 distinct values), but its training-validation gap of 0.433 shows the deep trees still follow the training years closely.

## 3.6. Support vector machine (pending)

## 3.7. Multilayer perceptron (pending)

## 3.8. Default scoreboard

In [53]:
default_runs = {**reference_runs, **nb_runs, **lda_runs, **knn_runs, **tree_runs, **forest_runs}
board_default = judge(default_runs, reference="logistic").sort_values("f1", ascending=False)
board_default.loc["dummy", "n_features"] = 0
display(board_default[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse",
                       "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
random_forest,38.000,0.736,-0.015,-0.029,-0.002,4,0.669,0.760,0.589,0.440,worse
hist_gb,38.000,0.736,-0.015,-0.031,0.001,4,0.654,0.760,0.607,0.300,no clear difference
lda_selected,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
lda,69.250,0.733,-0.018,-0.031,-0.006,3,0.645,0.741,0.616,0.350,worse
knn_selected,20.000,0.732,-0.018,-0.036,-0.001,3,0.643,0.731,2.130,0.300,worse
knn,69.250,0.726,-0.024,-0.041,-0.008,4,0.632,0.734,1.981,0.300,worse
dummy,0.000,0.709,-0.042,-0.059,-0.021,4,0.500,0.500,0.689,0.300,worse
naive_bayes_selected,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse


In [54]:
order = board_default.index[::-1]  # best at the top of the chart
fig = go.Figure(go.Scatter(
    x=board_default.loc[order, "f1_diff"], y=order, mode="markers",
    marker=dict(size=10, color=ACCENT["burgundy"]),
    error_x=dict(
        type="data", symmetric=False,
        array=(board_default.loc[order, "ci_high"] - board_default.loc[order, "f1_diff"]).fillna(0),
        arrayminus=(board_default.loc[order, "f1_diff"] - board_default.loc[order, "ci_low"]).fillna(0),
        color=ACCENT["near_black"], thickness=1.2,
    ),
    customdata=board_default.loc[order, ["f1", "ci_low", "ci_high"]],
    hovertemplate="%{y}<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                  "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<extra></extra>",
))
fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
fig.update_layout(
    title="Default models: F1 difference from logistic regression, 95% customer-bootstrap interval",
    xaxis_title="F1 minus logistic regression's F1",
    height=460,
)
fig.write_html(FIGURES_DIR / "04_default_scoreboard.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_default_scoreboard.png", **PNG_OPTIONS)
fig.show()

No default model beats logistic regression. Every interval lies below zero or touches it, and gradient boosting is the only "no clear difference" ([-0.031, 0.001]). Random forest, gradient boosting, LDA and KNN cluster at F1 0.726–0.736. Naive Bayes and the single tree fall below the always-repeat floor.

## Findings

| Model | F1 | Difference [95% interval] | ROC-AUC | Main issue at defaults |
|---|---|---|---|---|
| Logistic regression | 0.750 | reference | 0.784 | — |
| Random forest | 0.736 | -0.015 [-0.029, -0.002] | 0.760 | deep trees, gap 0.433 |
| Gradient boosting | 0.736 | -0.015 [-0.031, 0.001] | 0.760 | gap 0.348 |
| LDA (selected) | 0.733 | -0.017 [-0.030, -0.002] | 0.752 | singular covariance |
| KNN (selected) | 0.732 | -0.018 [-0.036, -0.001] | 0.731 | six probability values |
| Naive Bayes (selected) | 0.703 | -0.048 [-0.098, -0.019] | 0.756 | overconfident probabilities |
| Decision tree | 0.626 | -0.124 [-0.161, -0.092] | 0.592 | one-row leaves, 0/1 probabilities |

- At default settings, logistic regression stays the best model. Each other family fails in a way its check predicted: overconfidence for Naive Bayes, collinearity for LDA, coarse votes for KNN and overfitting for the trees.
- The elastic-net selection helped Naive Bayes and KNN and did not change LDA, so the selected versions go forward.
- Every failure has a matching setting to tune: calibration and smoothing, shrinkage, the number of neighbours, and pruning or depth limits.

# 4. Tuning

Each model family is tuned on the development folds with two Optuna searches of 60 trials each: one steered by log-loss and one by F1. Every trial records both. The preprocessing is fitted once per fold and reused by every trial, because only the model's settings change. The five best distinct trials of each search are rescored and compared with the family's default model under the decision rule.

## 4.0. Search setup

In [55]:
# Fingerprint of the saved inputs: a change in data or configuration starts new searches
fingerprint = hashlib.sha256(EXPERIMENT_VERSION.encode("utf-8"))
for filename in ["train_clean.parquet", "test_clean.parquet", "column_roles.json", "preprocessing_config.json"]:
    fingerprint.update((PROCESSED_DIR / filename).read_bytes())
INPUT_FINGERPRINT = fingerprint.hexdigest()

STORAGE = f"sqlite:///{(OPTUNA_DIR / f'studies_{EXPERIMENT_VERSION}_{INPUT_FINGERPRINT[:12]}.db').as_posix()}"
STUDY_METADATA = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "validation_years": [int(year) for year in VAL_YEARS],
    "gap_days": GAP_DAYS,
}
OBJECTIVES = {"log_loss": "minimize", "f1": "maximize"}


class FoldCache:
    """Preprocessing fitted once per fold on the training rows; matrices reused by every trial."""

    def __init__(self, prep, inputs, data=dev):
        self.folds = []
        for (tr, va), year in zip(folds, VAL_YEARS):
            train_part, val_part = data.iloc[tr], data.iloc[va]
            fitted = clone(prep).fit(train_part[inputs], train_part[target])
            self.folds.append({
                "year": year,
                "X_train": np.asarray(fitted.transform(train_part[inputs]), dtype=float),
                "y_train": train_part[target].to_numpy(),
                "X_val": np.asarray(fitted.transform(val_part[inputs]), dtype=float),
                "y_val": val_part[target].to_numpy(),
                "ID": val_part["ID"].to_numpy(),
                "customer_id": val_part["customer_id"].to_numpy(),
                "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(),
            })


def evaluate_cached(cache, model):
    """Same output as score_folds, on cached matrices."""
    rows, oof = [], []
    for fold in cache.folds:
        start = perf_counter()
        fitted = clone(model).fit(fold["X_train"], fold["y_train"])
        seconds = perf_counter() - start
        p = fitted.predict_proba(fold["X_val"])[:, 1]
        p_train = fitted.predict_proba(fold["X_train"])[:, 1]
        rows.append({
            "fold": fold["year"],
            "n_features": fold["X_train"].shape[1],
            "log_loss": log_loss(fold["y_val"], p, labels=[0, 1]),
            "train_log_loss": log_loss(fold["y_train"], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(fold["y_val"], p),
            "fit_seconds": seconds,
        })
        oof.append(pd.DataFrame({key: fold[key] for key in ["ID", "customer_id", "seen"]}
                                | {"fold": fold["year"], "y": fold["y_val"], "p": p}))

    oof = pd.concat(oof, ignore_index=True)[["fold", "ID", "customer_id", "seen", "y", "p"]]
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold

def pick_cache(cache, params):
    """A search can switch between caches (e.g. all or selected columns) through a function of its settings."""
    return cache(params) if callable(cache) else cache

def tune(cache, make_model, suggest, objective, name, n_trials=N_TRIALS):
    """Run or resume a stored Optuna search steered by log-loss or F1; every trial records both."""
    if FRESH_SEARCH:
        try:
            optuna.delete_study(study_name=name, storage=STORAGE)
        except KeyError:
            pass

    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True, direction=OBJECTIVES[objective],
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),  # same seed, same trials on a rerun
    )
    expected = {**STUDY_METADATA, "objective": objective}
    if study.trials and any(study.user_attrs.get(k) != v for k, v in expected.items()):
        raise ValueError(f"{name}: stored search was run with other settings. Use a new experiment version.")
    for key, value in expected.items():
        study.set_user_attr(key, value)

    def run_trial(trial):
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)  # a model that does not converge stops the search
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(pick_cache(cache, params), make_model(params))
        metrics = summary_row(table, oof, threshold)
        for key in ["f1", "log_loss", "roc_auc", "balanced_accuracy", "threshold", "overfit_gap"]:
            trial.set_user_attr(key, float(metrics[key]))
        return metrics[objective]

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < n_trials:
        study.optimize(run_trial, n_trials=n_trials - done, show_progress_bar=False)
    print(f"{name}: {len(study.trials)} trials")
    return study


def trials_table(study):
    """Completed trials, best first by the steering objective."""
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    table = pd.DataFrame([{"trial": t.number, **t.params, **t.user_attrs} for t in done]).set_index("trial")
    objective = study.user_attrs["objective"]
    return table.sort_values(objective, ascending=OBJECTIVES[objective] == "minimize", kind="stable")


def shortlist(studies, cache, make_model, k=5):
    """Rescore the k best trials of each search, named like 'f1_trial_12'."""
    runs = {}
    for objective, study in studies.items():
        # Trials with identical scores are the same model in practice; keep one of each
        distinct = trials_table(study).round({"f1": 4, "log_loss": 4}).drop_duplicates(subset=["f1", "log_loss"])
        for number in distinct.head(k).index:
            params = study.trials[number].params
            runs[f"{objective}_trial_{number}"] = (evaluate_cached(pick_cache(cache, params), make_model(params)), params)
    return runs


def choose_winner(runs, reference):
    """Keep the reference unless a candidate is better; among better ones, fewer columns then higher F1."""
    board = judge(runs, reference)
    better = board.index[board["verdict"] == "better"].tolist()
    if not better:
        return reference, board
    best = board.loc[better].sort_values(["n_features", "f1"], ascending=[True, False]).index[0]
    return best, board

### Tuned references and tracking

The tuned models are compared with the tuned references of notebook 3: logistic regression, which kept its default settings, and gradient boosting with its selected F1 trial. After each family is tuned, its chosen model joins one comparison against logistic regression, shown beside its default to make the effect of tuning visible.

In [56]:
# Tuned references from notebook 3: logistic regression kept its defaults, gradient boosting took F1 trial 44
hgb_params = next(d.carries["tree_params"] for d in log.items if d.step == "D33")
tuned_runs = {
    "logistic": reference_runs["logistic"],
    "hist_gb": score_folds(
        build_tree_pipeline(HistGradientBoostingClassifier(**hgb_params, random_state=RANDOM_STATE)), tree_inputs
    ),
}
# Each family's default run, to show what tuning added
default_of = {"logistic": reference_runs["logistic"], "hist_gb": reference_runs["hist_gb"]}


def plot_search(studies, model_name):
    """Search diagnostics, as in notebook 3: history, setting importance, and every trial's scores."""
    for objective, study in studies.items():
        plot_optimization_history(study).update_layout(
            title=f"{model_name}, steered by {objective}: search history").show()
        plot_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=RANDOM_STATE)).update_layout(
            title=f"{model_name}, steered by {objective}: which settings mattered").show()
    plot_slice(studies["log_loss"]).update_layout(title=f"{model_name}: log-loss against each setting").show()

    # Left: log-loss against F1 for every trial. Right: training against validation log-loss.
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Development performance", "Training and validation loss"])
    for i, (objective, study) in enumerate(studies.items()):
        trials = trials_table(study)
        train_loss = trials["log_loss"] - trials["overfit_gap"]
        style = dict(mode="markers", marker=dict(color=SERIES_COLOURS[i], size=7, opacity=0.7),
                     legendgroup=objective, name=f"steered by {objective}")
        fig.add_trace(go.Scatter(x=trials["log_loss"], y=trials["f1"], **style,
                                 hovertemplate="log-loss %{x:.3f}<br>F1 %{y:.3f}<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=train_loss, y=trials["log_loss"], **style, showlegend=False,
                                 hovertemplate="training %{x:.3f}<br>validation %{y:.3f}<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0.4, 0.8], y=[0.4, 0.8], mode="lines", line=dict(dash="dash", color="grey"),
                             name="equal losses"), row=1, col=2)
    fig.update_xaxes(title_text="Mean validation log-loss", row=1, col=1)
    fig.update_yaxes(title_text="Pooled F1", row=1, col=1)
    fig.update_xaxes(title_text="Mean training log-loss", row=1, col=2)
    fig.update_yaxes(title_text="Mean validation log-loss", row=1, col=2)
    fig.update_layout(title=f"{model_name}: every trial of both searches", height=450)
    fig.show()


def plot_tuning_progress():
    """Tuned families so far against logistic regression, with each family's default for contrast."""
    board = judge(tuned_runs, reference="logistic")
    ref_f1 = board.loc["logistic", "f1"]
    default_diff = pd.Series({name: summary_row(*run)["f1"] - ref_f1 for name, run in default_of.items()})
    order = board.sort_values("f1").index

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=default_diff[order], y=order, mode="markers", name="default",
        marker=dict(symbol="circle-open", size=11, color=ACCENT["near_black"]),
        hovertemplate="%{y} default<br>difference %{x:+.3f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=board.loc[order, "f1_diff"], y=order, mode="markers", name="tuned",
        marker=dict(size=11, color=ACCENT["burgundy"]),
        error_x=dict(type="data", symmetric=False,
                     array=(board.loc[order, "ci_high"] - board.loc[order, "f1_diff"]).fillna(0),
                     arrayminus=(board.loc[order, "f1_diff"] - board.loc[order, "ci_low"]).fillna(0),
                     color=ACCENT["burgundy"], thickness=1.2),
        customdata=board.loc[order, ["f1", "ci_low", "ci_high", "verdict"]],
        hovertemplate="%{y} tuned<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                      "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<br>%{customdata[3]}<extra></extra>",
    ))
    fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
    fig.update_layout(
        title=f"Tuned models against logistic regression ({len(board)} families so far)",
        xaxis_title="F1 minus logistic regression's F1 (bars: 95% customer-bootstrap interval)",
        height=150 + 55 * len(board),
    )
    fig.write_html(FIGURES_DIR / "04_tuning_progress.html", **HTML_OPTIONS)
    fig.write_image(FIGURES_DIR / "04_tuning_progress.png", **PNG_OPTIONS)
    fig.show()
    display(board[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "balanced_accuracy",
                   "roc_auc", "log_loss", "threshold", "verdict"]])

## 4.1. Naive Bayes

The default model's main problem was overconfidence. Two settings address it: `var_smoothing`, which adds a share of the largest variance to every variance, and calibration, which relearns the probabilities on held-out parts of the training years (sigmoid or isotonic). The elastic-net columns are kept.

In [57]:
# decision: Naive Bayes keeps the elastic-net selection (better than all columns at defaults)
nb_cache = FoldCache(Pipeline(build_linear_pipeline(GaussianNB()).steps[:-1]), linear_inputs)


def suggest_nb(trial):
    return {
        "var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 1.0, log=True),
        "calibration": trial.suggest_categorical("calibration", ["none", "sigmoid", "isotonic"]),
    }


def make_nb(params):
    model = GaussianNB(var_smoothing=params["var_smoothing"])
    if params["calibration"] == "none":
        return model
    # Calibration learns on 5 internal splits of the training years only
    return CalibratedClassifierCV(model, method=params["calibration"], cv=5)

In [58]:
study_nb = {objective: tune(nb_cache, make_nb, suggest_nb, objective, f"naive_bayes_{objective}")
            for objective in OBJECTIVES}

for objective, study in study_nb.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["var_smoothing", "calibration", "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"var_smoothing": "{:.1e}"}, precision=3))

naive_bayes_log_loss: 60 trials
naive_bayes_f1: 60 trials

Steered by log_loss


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
54,1.0e-12,isotonic,0.725,0.593,0.756,0.310
41,1.0e-12,isotonic,0.725,0.593,0.756,0.310
51,1.1e-12,isotonic,0.725,0.593,0.756,0.310
42,1.1e-12,isotonic,0.725,0.593,0.756,0.310
52,1.1e-12,isotonic,0.725,0.593,0.756,0.310



Steered by f1


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
53,3.2e-05,isotonic,0.727,0.593,0.756,0.320
54,6.1e-05,isotonic,0.727,0.593,0.756,0.320
55,5.2e-05,isotonic,0.727,0.593,0.756,0.320
1,7.5e-11,isotonic,0.725,0.593,0.756,0.310
2,1.6e-05,isotonic,0.725,0.593,0.756,0.310


In [59]:
plot_search(study_nb, "Naive Bayes")

Both searches settle on isotonic calibration with negligible smoothing (below 10⁻⁴). The best trials reach F1 0.725–0.727 and log-loss 0.593, against 0.703 and 2.568 for the default. Of the 60 trials of each search, 37–39 tried isotonic calibration and none of the uncalibrated ones passed F1 0.708.

In [60]:
nb_candidates = shortlist(study_nb, nb_cache, make_nb)
runs = {"default": nb_runs["naive_bayes_selected"], **{name: run for name, (run, _) in nb_candidates.items()}}
chosen_nb, board_nb_tuned = choose_winner(runs, reference="default")
nb_params = nb_candidates[chosen_nb][1] if chosen_nb != "default" else {}

display(board_nb_tuned[["f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_nb} {nb_params}")

,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,
default,0.703,0.000,NaN,NaN,0,2.568,0.756,reference
log_loss_trial_54,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference
log_loss_trial_7,0.711,0.008,-0.001,0.023,1,0.595,0.759,no clear difference
log_loss_trial_16,0.713,0.010,-0.023,0.064,2,0.604,0.746,no clear difference
log_loss_trial_24,0.720,0.018,-0.011,0.066,1,0.611,0.747,no clear difference
log_loss_trial_40,0.723,0.020,-0.009,0.069,1,0.616,0.756,no clear difference
f1_trial_53,0.727,0.024,-0.004,0.071,1,0.593,0.756,no clear difference
f1_trial_1,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference
f1_trial_36,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference


Chosen: default {}


In [61]:
tuned_runs["naive_bayes"] = runs[chosen_nb]
default_of["naive_bayes"] = nb_runs["naive_bayes_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse


Calibration raises F1 by about +0.024, but the customer-bootstrap interval [-0.004, 0.071] includes zero, so under the decision rule the default stays. ROC-AUC does not move (0.756), because isotonic calibration keeps the order of the predictions and only changes their scale.

In [62]:
# Reliability on the pooled development predictions: default against the best calibrated trial
best_calibrated = board_nb_tuned.drop(index="default").sort_values("f1", ascending=False).index[0]
curves = {"default (uncalibrated)": runs["default"][1], f"calibrated ({best_calibrated})": runs[best_calibrated][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
brier = {}
for name, oof in curves.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    brier[name] = brier_score_loss(oof["y"], oof["p"])
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Naive Bayes: predicted against observed repeat share (deciles of the predictions)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_naive_bayes_calibration.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_naive_bayes_calibration.png", **PNG_OPTIONS)
fig.show()

print({name: round(value, 3) for name, value in brier.items()})

{'default (uncalibrated)': 0.287, 'calibrated (f1_trial_53)': 0.203}


The uncalibrated model pushes almost every prediction to 0 or 1: its lower deciles all predict about 0.00 while 29–43% of those purchases are followed by a repeat. After calibration the points follow the diagonal, from about 0.19 predicted against 0.31 observed in the lowest decile to 0.96 against 0.95 in the highest. The Brier score falls from 0.287 to 0.203.

## 4.2. Linear discriminant analysis

The default LDA ran on a singular covariance matrix. Shrinkage pulls the estimated covariance towards a diagonal one, which makes it invertible and more stable. The search tries no shrinkage, the automatic Ledoit–Wolf amount and fixed amounts from 0.0001 to 1, on all prepared columns or on the elastic-net selection.

In [63]:
# Two cached input sets: all prepared columns, or the elastic-net selection
lda_caches = {
    "all": FoldCache(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(LinearDiscriminantAnalysis()).steps[:-1]), linear_inputs),
}


def suggest_lda(trial):
    params = {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "shrinkage_mode": trial.suggest_categorical("shrinkage_mode", ["none", "auto", "value"]),
    }
    if params["shrinkage_mode"] == "value":
        params["shrinkage"] = trial.suggest_float("shrinkage", 1e-4, 1.0, log=True)
    return params


def make_lda(params):
    if params["shrinkage_mode"] == "none":
        return LinearDiscriminantAnalysis(solver="svd")
    # Shrinkage needs a solver that estimates the covariance: lsqr
    shrinkage = "auto" if params["shrinkage_mode"] == "auto" else params["shrinkage"]
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=shrinkage)

In [64]:
get_lda_cache = lambda params: lda_caches[params["columns"]]

study_lda = {objective: tune(get_lda_cache, make_lda, suggest_lda, objective, f"lda_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_lda.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5).reindex(
        columns=["columns", "shrinkage_mode", "shrinkage", "f1", "log_loss", "roc_auc", "threshold"]
    )
    display(top.style.format({"shrinkage": "{:.3g}"}, precision=3, na_rep="-"))

lda_log_loss: 60 trials
lda_f1: 60 trials

Steered by log_loss


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
54,selected,value,0.113,0.735,0.586,0.759,0.310
55,selected,value,0.195,0.735,0.587,0.760,0.310
58,selected,value,0.202,0.735,0.587,0.760,0.310
59,selected,value,0.204,0.735,0.587,0.760,0.310
56,selected,value,0.218,0.736,0.587,0.760,0.310



Steered by f1


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
32,all,value,0.0211,0.740,0.594,0.756,0.350
33,all,value,0.021,0.740,0.594,0.756,0.350
36,all,value,0.021,0.740,0.594,0.756,0.350
50,all,value,0.0193,0.740,0.594,0.756,0.350
51,all,value,0.0187,0.739,0.594,0.756,0.350


In [65]:
plot_search(study_lda, "LDA")

The two searches find different optima. Log-loss prefers the selected columns with moderate shrinkage (about 0.1–0.2, log-loss 0.586–0.587). F1 prefers all columns with light shrinkage (about 0.02, F1 0.740). Every top trial uses a fixed shrinkage amount rather than none or the automatic one.

In [66]:
lda_candidates = shortlist(study_lda, get_lda_cache, make_lda)
runs = {"default": lda_runs["lda_selected"], **{name: run for name, (run, _) in lda_candidates.items()}}
chosen_lda, board_lda_tuned = choose_winner(runs, reference="default")
lda_params = lda_candidates[chosen_lda][1] if chosen_lda != "default" else {}

display(board_lda_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_lda} {lda_params}")

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,
default,20.000,0.733,0.000,NaN,NaN,0,0.593,0.752,reference
log_loss_trial_54,20.000,0.735,0.002,-0.006,0.009,3,0.586,0.759,no clear difference
log_loss_trial_55,20.000,0.735,0.002,-0.007,0.009,2,0.587,0.760,no clear difference
log_loss_trial_58,20.000,0.735,0.002,-0.008,0.009,2,0.587,0.760,no clear difference
log_loss_trial_59,20.000,0.735,0.002,-0.008,0.009,2,0.587,0.760,no clear difference
log_loss_trial_56,20.000,0.736,0.003,-0.006,0.011,2,0.587,0.760,no clear difference
f1_trial_32,69.250,0.740,0.007,-0.006,0.016,1,0.594,0.756,no clear difference
f1_trial_50,69.250,0.740,0.007,-0.006,0.016,1,0.594,0.756,no clear difference
f1_trial_51,69.250,0.739,0.006,-0.006,0.016,1,0.594,0.756,no clear difference


Chosen: default {}


In [67]:
tuned_runs["lda"] = runs[chosen_lda]
default_of["lda"] = lda_runs["lda_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse
lda,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse


No candidate passes the rule. The best gain in F1 is +0.007 (0.740 against 0.733), with an interval of [-0.006, 0.016] and worse in one year. The log-loss candidates gain +0.002 to +0.003 and are worse in two or three years. The default LDA on the selected columns stays.

In [68]:
# F1 and log-loss across the shrinkage values tried (both searches, fixed-value trials only)
tried = pd.concat([trials_table(study) for study in study_lda.values()])
tried = tried.loc[tried["shrinkage_mode"] == "value"].sort_values("shrinkage")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled F1", "Mean validation log-loss"])
for i, (columns, part) in enumerate(tried.groupby("columns")):
    for col, metric in enumerate(["f1", "log_loss"], start=1):
        fig.add_trace(go.Scatter(
            x=part["shrinkage"], y=part[metric], mode="markers", name=f"{columns} columns",
            marker=dict(color=SERIES_COLOURS[i], size=8), legendgroup=columns, showlegend=col == 1,
            hovertemplate="shrinkage %{x:.3g}<br>" + metric + " %{y:.3f}<extra>" + columns + "</extra>",
        ), row=1, col=col)
fig.update_xaxes(type="log", title="Shrinkage (log scale)")
fig.update_layout(title="LDA: effect of shrinkage on F1 and log-loss", height=430)
fig.write_html(FIGURES_DIR / "04_lda_shrinkage.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_lda_shrinkage.png", **PNG_OPTIONS)
fig.show()

F1 barely moves with shrinkage: 0.723–0.740 on all columns and 0.732–0.736 on the selected ones. Log-loss shows the trade-off more clearly. On all columns it falls from about 0.61 with almost no shrinkage to 0.589 around 0.1, then rises sharply towards 1.51 at full shrinkage, where the covariance becomes diagonal and the model ignores how the columns move together. The selected columns are less sensitive (0.586–0.625).

## 4.3. K-nearest neighbours

Four settings are searched: the input columns (all or the elastic-net selection), the number of neighbours k from 1 to 151, uniform or distance-weighted votes, and Manhattan (p = 1) or Euclidean (p = 2) distance. Manhattan distance adds absolute differences, so a single extreme column weighs less than with squared differences.

In [69]:
knn_caches = {
    "all": FoldCache(build_linear_pipeline(KNeighborsClassifier(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(KNeighborsClassifier()).steps[:-1]), linear_inputs),
}


def suggest_knn(trial):
    return {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 151, log=True),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "p": trial.suggest_categorical("p", [1, 2]),  # 1 Manhattan, 2 Euclidean
    }


def make_knn(params):
    return KNeighborsClassifier(n_neighbors=params["n_neighbors"], weights=params["weights"], p=params["p"])

In [70]:
get_knn_cache = lambda params: knn_caches[params["columns"]]

study_knn = {objective: tune(get_knn_cache, make_knn, suggest_knn, objective, f"knn_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_knn.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["columns", "n_neighbors", "weights", "p", "f1", "log_loss", "roc_auc", "threshold"]])

knn_log_loss: 60 trials
knn_f1: 60 trials

Steered by log_loss


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
51,selected,55,distance,1,0.741,0.566,0.778,0.330
42,selected,62,distance,1,0.741,0.566,0.778,0.320
44,selected,62,distance,1,0.741,0.566,0.778,0.320
52,selected,54,distance,1,0.738,0.566,0.778,0.330
22,selected,60,distance,1,0.739,0.567,0.778,0.310



Steered by f1


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
51,selected,110,distance,1,0.747,0.567,0.779,0.370
59,all,121,distance,1,0.747,0.570,0.778,0.400
30,selected,137,distance,1,0.746,0.571,0.776,0.370
40,all,150,distance,1,0.746,0.573,0.775,0.400
39,all,143,distance,1,0.746,0.572,0.776,0.390


In [71]:
plot_search(study_knn, "KNN")

Both searches converge on distance-weighted votes with Manhattan distance. They differ on k: log-loss prefers about 55–62 neighbours (log-loss 0.566, F1 0.741), while F1 prefers 110–150 (F1 0.746–0.747). ROC-AUC rises from 0.731 at the default to 0.778–0.779.

In [72]:
knn_candidates = shortlist(study_knn, get_knn_cache, make_knn)
runs = {"default": knn_runs["knn_selected"], **{name: run for name, (run, _) in knn_candidates.items()}}
chosen_knn, board_knn_tuned = choose_winner(runs, reference="default")
knn_params = knn_candidates[chosen_knn][1] if chosen_knn != "default" else {}

display(board_knn_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_knn} {knn_params}")

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,
default,20.000,0.732,0.000,NaN,NaN,0,2.130,0.731,reference
log_loss_trial_51,20.000,0.741,0.009,-0.004,0.023,0,0.566,0.778,no clear difference
log_loss_trial_42,20.000,0.741,0.008,-0.004,0.024,0,0.566,0.778,no clear difference
log_loss_trial_52,20.000,0.738,0.006,-0.007,0.020,0,0.566,0.778,no clear difference
log_loss_trial_22,20.000,0.739,0.007,-0.006,0.023,0,0.567,0.778,no clear difference
log_loss_trial_21,20.000,0.739,0.006,-0.007,0.022,1,0.567,0.778,no clear difference
f1_trial_51,20.000,0.747,0.015,0.001,0.030,0,0.567,0.779,better
f1_trial_59,69.250,0.747,0.015,-0.001,0.030,0,0.570,0.778,no clear difference
f1_trial_30,20.000,0.746,0.014,0.000,0.028,0,0.571,0.776,better


Chosen: f1_trial_51 {'columns': 'selected', 'n_neighbors': 110, 'weights': 'distance', 'p': 1}


In [73]:
tuned_runs["knn"] = runs[chosen_knn]
default_of["knn"] = knn_runs["knn_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse
lda,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
knn,20.000,0.747,-0.003,-0.017,0.011,2,0.668,0.779,0.567,0.370,no clear difference


Two F1-search trials pass the rule against the default: trial 51 (110 neighbours, F1 0.747, interval [0.001, 0.030]) and trial 30 (137 neighbours, [0.000, 0.028]), both on the 20 selected columns and worse in no year. Trial 51 is chosen as the higher F1 at the same column count. The log-loss candidates gain +0.006 to +0.009 with intervals that include zero.

In [74]:
# Elbow: F1 against k for both weightings, columns and distance of the chosen trial held fixed
elbow_cache = knn_caches[knn_params["columns"]]
elbow_rows = []
for weights in ["uniform", "distance"]:
    for k in [1, 3, 5, 9, 15, 25, 41, 61, 81, 110, 151]:
        model = KNeighborsClassifier(n_neighbors=k, weights=weights, p=knn_params["p"])
        table, oof, threshold = evaluate_cached(elbow_cache, model)
        # Training F1 in each fold, at the same pooled threshold
        train_f1 = np.mean([
            f1_score(fold["y_train"], clone(model).fit(fold["X_train"], fold["y_train"])
                     .predict_proba(fold["X_train"])[:, 1] >= threshold)
            for fold in elbow_cache.folds
        ])
        elbow_rows.append({"weights": weights, "k": k, "train_f1": train_f1,
                           "validation_f1": f1_score(oof["y"], oof["p"] >= threshold)})
elbow = pd.DataFrame(elbow_rows)

fig = go.Figure()
for i, (weights, part) in enumerate(elbow.groupby("weights", sort=False)):
    fig.add_trace(go.Scatter(x=part["k"], y=part["validation_f1"], mode="lines+markers",
                             name=f"validation, {weights}", line=dict(color=SERIES_COLOURS[i]),
                             hovertemplate="k = %{x}<br>F1 %{y:.3f}<extra>validation, " + weights + "</extra>"))
    fig.add_trace(go.Scatter(x=part["k"], y=part["train_f1"], mode="lines", name=f"training, {weights}",
                             line=dict(color=SERIES_COLOURS[i], dash="dot"),
                             hovertemplate="k = %{x}<br>F1 %{y:.3f}<extra>training, " + weights + "</extra>"))
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"),
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_layout(
    title=f"KNN elbow: F1 against the number of neighbours (p = {knn_params['p']}, {knn_params['columns']} columns)",
    xaxis=dict(title="Number of neighbours k (log scale)", type="log", tickvals=sorted(elbow["k"].unique())),
    yaxis_title="F1",
    height=470,
)
fig.write_html(FIGURES_DIR / "04_knn_elbow.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_knn_elbow.png", **PNG_OPTIONS)
fig.show()

display(elbow.pivot(index="k", columns="weights", values=["train_f1", "validation_f1"]).round(3))

train_f1         validation_f1        
weights distance uniform      distance uniform
k                                             
1          1.000   1.000         0.668   0.668
3          1.000   0.802         0.732   0.732
5          1.000   0.804         0.731   0.731
9          1.000   0.779         0.734   0.732
15         1.000   0.769         0.735   0.733
25         1.000   0.757         0.741   0.740
41         1.000   0.755         0.743   0.742
61         1.000   0.752         0.739   0.740
81         1.000   0.748         0.743   0.745
110        1.000   0.749         0.747   0.746
151        1.000   0.747         0.743   0.742

With uniform votes the classic elbow appears: training F1 falls from 1.000 at k = 1 to 0.747 at 151, while validation F1 rises from 0.668 to about 0.740 by k = 25 and then stays between 0.740 and 0.746. The elbow sits around k = 25; larger k adds at most +0.006. With distance weighting, training F1 is 1.000 at every k, because each training row is its own nearest neighbour at distance zero, so only its validation curve is informative. That curve follows the uniform one closely.

## 4.4. Decision tree

The default tree grew until almost every leaf was pure and fell below the floor. Three settings limit its growth: `max_depth` caps the number of questions, `min_samples_leaf` sets the smallest group that can answer, and `ccp_alpha` removes branches whose impurity reduction is worth less than alpha per leaf (cost-complexity pruning). Before the search, the pruning path and a depth curve show how size and performance move together.

### Pruning path

In [75]:
# Missing values stay in the tree branch; preprocessing is cached once per fold
tree_cache = FoldCache(build_tree_pipeline(DecisionTreeClassifier()).named_steps["prep"], tree_inputs)

# Cost-complexity pruning path of the full tree in the last fold: the alphas at which branches disappear
last = tree_cache.folds[-1]
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(last["X_train"], last["y_train"])
# Log-spaced, so small and large prunings are both visible
alphas = np.geomspace(1e-5, path.ccp_alphas[:-1].max(), 30)

path_rows = []
for alpha in alphas:
    table, oof, threshold = evaluate_cached(tree_cache, DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE))
    leaves = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE).fit(
        last["X_train"], last["y_train"]).get_n_leaves()
    path_rows.append({"ccp_alpha": alpha, "leaves_last_fold": leaves,
                      "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                      "log_loss": table["log_loss"].mean()})
pruning = pd.DataFrame(path_rows)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["leaves_last_fold"], mode="lines+markers",
                         name="leaves (last fold)", line=dict(color=ACCENT["near_black"]),
                         hovertemplate="alpha %{x:.2e}<br>%{y} leaves<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["validation_f1"], mode="lines+markers",
                         name="validation F1 (pooled)", line=dict(color=ACCENT["burgundy"]),
                         hovertemplate="alpha %{x:.2e}<br>F1 %{y:.3f}<extra></extra>"), secondary_y=True)
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"), secondary_y=True,
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_xaxes(type="log", title="ccp_alpha (log scale)")
fig.update_yaxes(title_text="Leaves", type="log", secondary_y=False)
fig.update_yaxes(title_text="F1", secondary_y=True)
fig.update_layout(title="Decision tree: pruning path, tree size and validation F1", height=450)
fig.write_html(FIGURES_DIR / "04_tree_pruning_path.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_tree_pruning_path.png", **PNG_OPTIONS)
fig.show()

best_alpha = pruning.loc[pruning["validation_f1"].idxmax()]
print(f"Path alphas: {path.ccp_alphas.min():.1e} to {path.ccp_alphas[:-1].max():.1e}")
print(f"Best on the path: alpha {best_alpha['ccp_alpha']:.1e}, {best_alpha['leaves_last_fold']:.0f} leaves, "
      f"F1 {best_alpha['validation_f1']:.3f}")

Path alphas: 0.0e+00 to 1.3e-02
Best on the path: alpha 2.2e-03, 8 leaves, F1 0.732


Below an alpha of about 10⁻⁴ nothing is pruned: the tree keeps 830 leaves in the last fold and F1 stays at 0.626. Between 10⁻⁴ and 2 × 10⁻³ the tree shrinks from about 800 leaves to 8, while F1 rises to 0.732 and log-loss falls from 14.7 to 0.567. Beyond that, trees of 2–6 leaves hold F1 between 0.720 and 0.729: most of the gain comes from removing the hundreds of small leaves.

### Depth curve

In [76]:
# Training against validation F1 as the tree is allowed to grow deeper
depth_rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    table, oof, threshold = evaluate_cached(tree_cache, model)
    train_f1 = np.mean([
        f1_score(fold["y_train"], clone(model).fit(fold["X_train"], fold["y_train"])
                 .predict_proba(fold["X_train"])[:, 1] >= threshold)
        for fold in tree_cache.folds
    ])
    depth_rows.append({"max_depth": "none" if depth is None else str(depth), "train_f1": train_f1,
                       "validation_f1": f1_score(oof["y"], oof["p"] >= threshold)})
depth_curve = pd.DataFrame(depth_rows).set_index("max_depth")

fig = go.Figure()
for column, name in [("train_f1", "training (mean of folds)"), ("validation_f1", "validation (pooled)")]:
    fig.add_trace(go.Scatter(x=depth_curve.index, y=depth_curve[column], mode="lines+markers", name=name,
                             hovertemplate="max_depth %{x}<br>F1 %{y:.3f}<extra>" + name + "</extra>"))
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"),
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_layout(title="Decision tree: training and validation F1 against maximum depth",
                  xaxis_title="max_depth", yaxis_title="F1", height=430)
fig.write_html(FIGURES_DIR / "04_tree_depth_curve.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_tree_depth_curve.png", **PNG_OPTIONS)
fig.show()

display(depth_curve.round(3).T)

max_depth,1,2,3,4,5,6,8,10,12,15,20,none
train_f1,0.716,0.709,0.736,0.730,0.754,0.768,0.807,0.792,0.908,0.956,0.988,1.000
validation_f1,0.709,0.729,0.716,0.719,0.719,0.697,0.667,0.657,0.635,0.624,0.639,0.626


Training F1 climbs from 0.709 at depth 2 to 1.000 for the unlimited tree, while validation F1 peaks at 0.729 at depth 2 and falls steadily after depth 5, to 0.626. From depth 6 on, the two curves move apart: each extra level fits the training years more closely and the later year less well.

### Search

In [77]:
def suggest_tree(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 300, log=True),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 0.05, log=True),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced"]),
    }


def make_tree(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else "balanced"}
    return DecisionTreeClassifier(**settings, random_state=RANDOM_STATE)


study_tree = {objective: tune(tree_cache, make_tree, suggest_tree, objective, f"decision_tree_{objective}")
              for objective in OBJECTIVES}

for objective, study in study_tree.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["max_depth", "min_samples_leaf", "ccp_alpha", "criterion", "class_weight",
                                       "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"ccp_alpha": "{:.1e}"}, precision=3))

decision_tree_log_loss: 60 trials
decision_tree_f1: 60 trials

Steered by log_loss


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
29,7,45,2.7e-03,gini,balanced,0.731,0.562,0.763,0.380
30,7,59,2.7e-03,gini,balanced,0.731,0.562,0.763,0.380
55,10,25,2.2e-03,gini,balanced,0.727,0.562,0.764,0.280
32,6,98,1.5e-03,gini,balanced,0.735,0.563,0.764,0.340
59,11,39,3.2e-03,gini,balanced,0.724,0.563,0.760,0.340



Steered by f1


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
52,8,35,1.8e-03,entropy,none,0.736,0.642,0.756,0.400
57,9,69,2.5e-03,entropy,none,0.735,0.580,0.764,0.450
49,7,34,1.8e-03,entropy,none,0.734,0.638,0.758,0.410
18,17,35,1.8e-03,entropy,none,0.734,0.643,0.754,0.400
20,9,35,2.5e-03,entropy,none,0.734,0.630,0.763,0.440


In [78]:
plot_search(study_tree, "Decision tree")

Both searches keep trees of depth 6–11 with at least 25 rows per leaf and alpha around 2–3 × 10⁻³, the region the pruning path pointed to. They differ in the split criterion and the class weights: log-loss prefers Gini with balanced weights (log-loss 0.562), and F1 prefers entropy without weights (F1 0.734–0.736, but log-loss 0.58–0.64).

In [79]:
tree_candidates = shortlist(study_tree, tree_cache, make_tree)
runs = {"default": tree_runs["decision_tree"], **{name: run for name, (run, _) in tree_candidates.items()}}
chosen_tree, board_tree_tuned = choose_winner(runs, reference="default")
tree_params = tree_candidates[chosen_tree][1] if chosen_tree != "default" else {}

display(board_tree_tuned[["f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_tree} {tree_params}")

# Size of the chosen tree in the last fold
chosen_fit = make_tree(tree_params).fit(last["X_train"], last["y_train"])
print(f"Chosen tree in the last fold: depth {chosen_fit.get_depth()}, {chosen_fit.get_n_leaves()} leaves")

,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,
default,0.626,0.000,NaN,NaN,0,14.692,0.592,reference
log_loss_trial_29,0.731,0.105,0.077,0.141,0,0.562,0.763,better
log_loss_trial_55,0.727,0.101,0.071,0.139,0,0.562,0.764,better
log_loss_trial_32,0.735,0.108,0.079,0.146,0,0.563,0.764,better
log_loss_trial_59,0.724,0.098,0.069,0.134,0,0.563,0.760,better
log_loss_trial_25,0.726,0.100,0.071,0.137,0,0.563,0.760,better
f1_trial_52,0.736,0.110,0.080,0.145,0,0.642,0.756,better
f1_trial_57,0.735,0.109,0.080,0.143,0,0.580,0.764,better
f1_trial_49,0.734,0.108,0.079,0.143,0,0.638,0.758,better


Chosen: f1_trial_52 {'max_depth': 8, 'min_samples_leaf': 35, 'ccp_alpha': 0.001815682899367124, 'criterion': 'entropy', 'class_weight': 'none'}
Chosen tree in the last fold: depth 8, 21 leaves


All ten candidates are clearly better than the default tree: they gain +0.098 to +0.110 F1, and every interval lies well above zero. The highest F1 is trial 52 (depth 8, at least 35 rows per leaf, alpha 1.8 × 10⁻³, entropy), at F1 0.736. In the last fold it has 21 leaves, against 830 for the default.

In [80]:
tuned_runs["decision_tree"] = runs[chosen_tree]
default_of["decision_tree"] = tree_runs["decision_tree"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse
lda,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
knn,20.000,0.747,-0.003,-0.017,0.011,2,0.668,0.779,0.567,0.370,no clear difference
decision_tree,38.000,0.736,-0.015,-0.030,-0.001,4,0.657,0.756,0.642,0.400,worse


Tuning moves the tree from F1 0.626 to 0.736, the largest gain of any family, but it remains clearly worse than logistic regression (interval [-0.030, -0.001], worse in all four years). After five families, gradient boosting (0.753) and KNN (0.747) are the only tuned models whose difference from logistic regression is unclear; LDA, the tree and Naive Bayes are clearly worse.

## 4.5. Random forest

The default forest grew 100 unlimited trees with √p columns per split, and it followed the training years closely (log-loss gap 0.433). The search varies the number of trees, the share of columns tried at each split, the minimum leaf size, the depth limit and the class weights. Averaging more trees smooths the probabilities, while fewer columns per split make the trees more different from one another.

In [81]:
# Same cached tree-branch matrices as the single tree
forest_cache = tree_cache


def suggest_forest(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 100, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced", "balanced_subsample"]),
    }


def make_forest(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else params["class_weight"]}
    return RandomForestClassifier(**settings, random_state=RANDOM_STATE, n_jobs=-1)


study_forest = {objective: tune(forest_cache, make_forest, suggest_forest, objective, f"random_forest_{objective}")
                for objective in OBJECTIVES}

for objective, study in study_forest.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["n_estimators", "max_features", "min_samples_leaf", "max_depth",
                                         "class_weight", "f1", "log_loss", "roc_auc", "threshold"]])

random_forest_log_loss: 60 trials
random_forest_f1: 60 trials

Steered by log_loss


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
33,500,0.728,51,17,none,0.748,0.549,0.786,0.410
45,450,0.830,51,16,none,0.751,0.549,0.786,0.410
49,450,0.861,54,12,none,0.747,0.549,0.785,0.420
46,500,0.825,59,23,none,0.748,0.549,0.785,0.400
50,450,0.931,56,16,none,0.748,0.549,0.785,0.410



Steered by f1


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
43,250,0.681,22,26,none,0.751,0.551,0.786,0.410
21,100,0.223,10,19,none,0.751,0.552,0.786,0.450
0,250,0.956,24,19,none,0.751,0.550,0.786,0.400
51,200,0.636,43,24,none,0.751,0.549,0.786,0.420
11,250,0.515,14,27,none,0.751,0.553,0.784,0.420


Both searches settle on forests with unweighted classes (53 and 52 of 60 trials), many columns per split (`max_features` mostly 0.5–0.95) and large leaves (10–59 rows). The log-loss search prefers 450–500 trees and reaches log-loss 0.549; the F1 search reaches F1 0.751 with 100–250 trees. Every top trial has ROC-AUC 0.784–0.786, against 0.760 for the default forest.

In [82]:
plot_search(study_forest, "Random forest")

In [83]:
forest_candidates = shortlist(study_forest, forest_cache, make_forest)
runs = {"default": forest_runs["random_forest"], **{name: run for name, (run, _) in forest_candidates.items()}}
chosen_forest, board_forest_tuned = choose_winner(runs, reference="default")
forest_params = forest_candidates[chosen_forest][1] if chosen_forest != "default" else {}

display(board_forest_tuned[["f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_forest} {forest_params}")

,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,
default,0.736,0.000,NaN,NaN,0,0.589,0.760,reference
log_loss_trial_33,0.748,0.012,0.001,0.027,1,0.549,0.786,better
log_loss_trial_45,0.751,0.015,0.003,0.031,1,0.549,0.786,better
log_loss_trial_49,0.747,0.011,0.000,0.025,1,0.549,0.785,better
log_loss_trial_46,0.748,0.012,0.000,0.028,1,0.549,0.785,better
log_loss_trial_50,0.748,0.012,0.000,0.027,1,0.549,0.785,better
f1_trial_43,0.751,0.016,0.005,0.030,0,0.551,0.786,better
f1_trial_21,0.751,0.015,0.005,0.028,0,0.552,0.786,better
f1_trial_0,0.751,0.015,0.004,0.030,1,0.550,0.786,better


Chosen: f1_trial_43 {'n_estimators': 250, 'max_features': 0.6809087418528421, 'min_samples_leaf': 22, 'max_depth': 26, 'class_weight': 'none'}


All ten candidates are clearly better than the default forest, with gains of +0.011 to +0.016 F1 and every interval above zero. Trial 43 is chosen as the highest F1 at the same column count: 250 trees, 68% of the columns per split, at least 22 rows per leaf and depth up to 26. It scores F1 0.751 and log-loss 0.551 against 0.736 and 0.589 for the default, and is worse in no year.

In [84]:
# Number of trees: the chosen settings with more or fewer trees
settings = forest_params or {"max_features": "sqrt", "min_samples_leaf": 1, "max_depth": None, "class_weight": "none"}
tree_count_rows = []
for n_trees in [10, 25, 50, 100, 200, 300, 500]:
    table, oof, threshold = evaluate_cached(forest_cache, make_forest({**settings, "n_estimators": n_trees}))
    tree_count_rows.append({"n_estimators": n_trees, "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                            "log_loss": table["log_loss"].mean(), "roc_auc": table["roc_auc"].mean()})
tree_counts = pd.DataFrame(tree_count_rows).set_index("n_estimators")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled validation F1", "Mean validation log-loss"])
for col, metric in enumerate(["validation_f1", "log_loss"], start=1):
    fig.add_trace(go.Scatter(x=tree_counts.index, y=tree_counts[metric], mode="lines+markers", showlegend=False,
                             line=dict(color=ACCENT["burgundy"]),
                             hovertemplate="%{x} trees<br>%{y:.3f}<extra></extra>"), row=1, col=col)
fig.update_xaxes(type="log", title="Number of trees (log scale)", tickvals=list(tree_counts.index))
fig.update_layout(title="Random forest: effect of the number of trees (chosen settings)", height=420)
fig.write_html(FIGURES_DIR / "04_forest_tree_count.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_forest_tree_count.png", **PNG_OPTIONS)
fig.show()

display(tree_counts.round(3).T)

n_estimators,10,25,50,100,200,300,500
validation_f1,0.746,0.747,0.749,0.750,0.751,0.750,0.752
log_loss,0.576,0.557,0.553,0.552,0.551,0.551,0.551
roc_auc,0.772,0.778,0.783,0.785,0.786,0.786,0.785


F1 is almost flat in the number of trees: 0.746 with 10 trees and 0.750–0.752 from 100 onwards. Log-loss and ROC-AUC improve until about 100–200 trees (log-loss 0.576 to 0.551, ROC-AUC 0.772 to 0.786) and then stop moving. More trees mostly smooth the probabilities; beyond 200 they only add fitting time.

In [85]:
tuned_runs["random_forest"] = runs[chosen_forest]
default_of["random_forest"] = forest_runs["random_forest"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse
lda,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
knn,20.000,0.747,-0.003,-0.017,0.011,2,0.668,0.779,0.567,0.370,no clear difference
decision_tree,38.000,0.736,-0.015,-0.030,-0.001,4,0.657,0.756,0.642,0.400,worse
random_forest,38.000,0.751,0.001,-0.009,0.012,1,0.673,0.786,0.551,0.410,no clear difference


With all six families tuned, three are not clearly different from logistic regression: gradient boosting (F1 0.753, interval [-0.008, 0.015]), random forest (0.751, [-0.009, 0.012]) and KNN (0.747, [-0.017, 0.011]). LDA, the decision tree and Naive Bayes stay clearly worse. Random forest has the highest ROC-AUC (0.786) and the lowest log-loss (0.551), but no tuned model passes the improvement rule against logistic regression.

## Findings

| Family | Default F1 | Tuned F1 | Tuning kept? | Against logistic regression |
|---|---|---|---|---|
| Logistic regression | 0.750 | 0.750 | default (notebook 3) | reference |
| Gradient boosting | 0.736 | 0.753 | yes (notebook 3) | no clear difference |
| Random forest | 0.736 | 0.751 | yes | no clear difference |
| KNN | 0.732 | 0.747 | yes | no clear difference |
| LDA | 0.733 | 0.733 | no | worse |
| Decision tree | 0.626 | 0.736 | yes | worse |
| Naive Bayes | 0.703 | 0.703 | no | worse |

- Tuning paid off most where the default overfitted. The tree gained +0.110 by pruning to about 21 leaves, and the forest and KNN gained about +0.015 by averaging over larger groups (leaves of 22+ rows, 110 neighbours).
- Calibration fixed Naive Bayes's probabilities (log-loss 2.568 to 0.593) and shrinkage stabilised LDA, but neither gain in F1 passed the rule.
- Three tuned models now match logistic regression within the bootstrap uncertainty, and none beats it. They differ in how they reach similar F1: a linear boundary, boosted shallow trees, averaged deep trees and local neighbourhoods. That makes them candidates for a blend.

# 5. Scoreboard and model choice

## 5.1. Final scoreboard

Every family enters with its chosen setting. Finalists are the models that are not clearly worse than logistic regression under the decision rule.

In [87]:
board_final = judge({"dummy": reference_runs["dummy"], **tuned_runs}, reference="logistic").sort_values("f1", ascending=False)
board_final.loc["dummy", "n_features"] = 0
display(board_final[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

# decision: finalists are the models not clearly worse than logistic regression
finalists = board_final.index[board_final["verdict"].isin(["reference", "no clear difference"])].tolist()
print("Finalists:", finalists)

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
hist_gb,38.000,0.753,0.003,-0.008,0.015,1,0.683,0.784,0.562,0.440,no clear difference
random_forest,38.000,0.751,0.001,-0.009,0.012,1,0.673,0.786,0.551,0.410,no clear difference
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
knn,20.000,0.747,-0.003,-0.017,0.011,2,0.668,0.779,0.567,0.370,no clear difference
decision_tree,38.000,0.736,-0.015,-0.030,-0.001,4,0.657,0.756,0.642,0.400,worse
lda,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
dummy,0.000,0.709,-0.042,-0.059,-0.021,4,0.500,0.500,0.689,0.300,worse
naive_bayes,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse


Finalists: ['hist_gb', 'random_forest', 'logistic', 'knn']


Four models are finalists: gradient boosting (F1 0.753), random forest (0.751), logistic regression (0.750) and KNN (0.747). They lie within 0.006 of one another, and every interval against logistic regression includes zero. Random forest has the best ROC-AUC (0.786) and log-loss (0.551); logistic regression has the best balanced accuracy among them (0.684) with the fewest columns (20).

In [88]:
fold_f1_final = pd.DataFrame({name: tuned_runs[name][0]["f1"] for name in finalists})
fold_f1_final["always repeat"] = [f1_floor(part["y"]) for _, part in tuned_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in fold_f1_final:
    floor = name == "always repeat"
    fig.add_trace(go.Scatter(
        x=fold_f1_final.index, y=fold_f1_final[name], mode="lines" if floor else "lines+markers", name=name,
        line=dict(dash="dash", color="grey") if floor else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Finalists: F1 in each validation year (tuned settings)",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis_title="F1 at each model's pooled threshold",
    height=430,
)
fig.write_html(FIGURES_DIR / "04_finalists_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_f1_by_year.png", **PNG_OPTIONS)
fig.show()

display(fold_f1_final.round(3))

,hist_gb,random_forest,logistic,knn,always repeat
fold,,,,,
2020,0.758,0.750,0.758,0.768,0.745
2021,0.767,0.766,0.764,0.741,0.709
2022,0.723,0.719,0.716,0.704,0.649
2023,0.760,0.768,0.760,0.777,0.728


No finalist leads in every year. KNN is the best in 2020 (0.768) and 2023 (0.777) but the weakest in 2021 (0.741) and 2022 (0.704). The other three stay within 0.009 of each other in every year. All four fall to their lowest F1 in 2022, the year with the lowest floor (0.649).

## 5.2. Blends

A blend averages the probabilities of several models (soft voting). It can only help where the members disagree, so their agreement is checked first.

In [89]:
# How similar are the finalists' predictions? Blends gain only where models disagree.
probabilities = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() for name in finalists})
labels = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() >= tuned_runs[name][2] for name in finalists})
agreement = pd.DataFrame(
    [[(labels[a] == labels[b]).mean() for b in finalists] for a in finalists], index=finalists, columns=finalists
)
correlation = probabilities.corr(method="spearman")

fig = go.Figure(go.Heatmap(
    z=correlation.to_numpy(), x=finalists, y=finalists, zmin=-1, zmax=1, colorscale=DIVERGING,
    text=correlation.round(2).to_numpy(), texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>rank correlation %{z:.3f}<extra></extra>",
))
fig.update_layout(title="Finalists: rank correlation of out-of-fold probabilities", height=430)
fig.write_html(FIGURES_DIR / "04_finalists_correlation.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_correlation.png", **PNG_OPTIONS)
fig.show()

print("Share of rows with the same predicted label:")
display(agreement.round(3))

Share of rows with the same predicted label:


,hist_gb,random_forest,logistic,knn
hist_gb,1.000,0.945,0.878,0.847
random_forest,0.945,1.000,0.878,0.851
logistic,0.878,0.878,1.000,0.857
knn,0.847,0.851,0.857,1.000


The two tree ensembles agree on 94.5% of the labels, so they carry largely the same information. Logistic regression agrees with each of them on 87.8% of the rows and with KNN on 85.7%, and KNN is the most different from the tree models (84.7–85.1%). Blending logistic regression with a tree model or with KNN therefore has more room to help than blending the two ensembles with each other.

In [90]:
def blend(names, weights=None):
    """Soft vote: weighted mean of the members' probabilities, with its own pooled threshold."""
    weights = np.ones(len(names)) / len(names) if weights is None else np.asarray(weights) / np.sum(weights)
    base = tuned_runs[names[0]][1]
    for name in names[1:]:
        assert tuned_runs[name][1]["ID"].equals(base["ID"]), "Members must be scored on the same rows."

    oof = base[["fold", "ID", "customer_id", "seen", "y"]].copy()
    oof["p"] = np.column_stack([tuned_runs[name][1]["p"] for name in names]) @ weights
    threshold, _, _ = best_threshold(oof["y"], oof["p"])

    rows = []
    for year, part in oof.groupby("fold"):
        rows.append({
            "fold": year,
            "n_features": sum(tuned_runs[name][0].loc[year, "n_features"] for name in names),  # total member inputs
            "log_loss": log_loss(part["y"], part["p"], labels=[0, 1]),
            "train_log_loss": np.nan,  # members were fitted separately
            "roc_auc": roc_auc_score(part["y"], part["p"]),
            "fit_seconds": sum(tuned_runs[name][0].loc[year, "fit_seconds"] for name in names),
            "f1": f1_score(part["y"], part["p"] >= threshold),
        })
    return pd.DataFrame(rows).set_index("fold"), oof, threshold


# decision: equal weights only; fitting weights on the same folds would add another layer of selection
blend_members = {
    "logistic + forest": ["logistic", "random_forest"],
    "logistic + boosting": ["logistic", "hist_gb"],
    "logistic + knn": ["logistic", "knn"],
    "logistic + forest + boosting": ["logistic", "random_forest", "hist_gb"],
    "all finalists": finalists,
}
blend_runs = {name: blend(members) for name, members in blend_members.items()}

board_blend = judge({"logistic": tuned_runs["logistic"], **blend_runs}, reference="logistic")
display(board_blend[["f1", "f1_diff", "ci_low", "ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,
logistic,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
logistic + forest,0.753,0.003,-0.004,0.011,1,0.685,0.790,0.544,0.380,no clear difference
logistic + boosting,0.755,0.004,-0.003,0.013,1,0.682,0.789,0.548,0.380,no clear difference
logistic + knn,0.752,0.001,-0.007,0.009,1,0.691,0.788,0.552,0.370,no clear difference
logistic + forest + boosting,0.754,0.004,-0.004,0.014,1,0.679,0.789,0.547,0.390,no clear difference
all finalists,0.755,0.004,-0.005,0.013,1,0.694,0.791,0.546,0.410,no clear difference


Every blend improves the probabilities: log-loss falls from 0.561 to 0.544–0.552, and ROC-AUC rises from 0.784 to 0.788–0.791. F1 rises only slightly, to 0.752–0.755, and no blend passes the rule: the largest gain, +0.004 for "logistic + boosting" and for all four finalists, has intervals of [-0.003, 0.013] and [-0.005, 0.013]. Every blend is worse than logistic regression in one of the four years.

The rank correlations tell the same story: 0.987 between the two ensembles, 0.919–0.920 between logistic regression and each ensemble, and 0.899–0.904 for KNN with the others.

## 5.3. Time-respecting nested check

The tuned scores above come from searches that saw all four validation years and then were scored on the same years, so they carry some optimism. Here each family is tuned again using only the years before each validation year, and then scored on that year once. The threshold also comes from the earlier years only. The question is whether tuning still beats the default, and still matches logistic regression, when the search cannot see the year it is scored on. 2020 has no earlier year, so the outer years are 2021–2023.

In [93]:
NESTED_TRIALS = 20  # per outer year and family; smaller than the main searches to keep the run time reasonable


class FoldSubset:
    """The same cached matrices, restricted to some validation years."""

    def __init__(self, cache, positions):
        self.folds = [cache.folds[i] for i in positions]


def suggest_hgb(trial):
    # Same search space as notebook 3
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 50, 800, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 64, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 300, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 30, log=True),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }


def make_hgb(params):
    return HistGradientBoostingClassifier(
        **params, categorical_features=list(range(len(branch_roles("tree")["low"]))),
        early_stopping=False, random_state=RANDOM_STATE,
    )


# family: (cache, suggest, make, default model)
nested_families = {
    "hist_gb": (tree_cache, suggest_hgb, make_hgb, make_hgb({})),
    "random_forest": (tree_cache, suggest_forest, make_forest,
                      RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    "knn": (lambda params: knn_caches[params["columns"]], suggest_knn, make_knn, KNeighborsClassifier()),
}


def outer_f1(cache, model, inner, outer):
    """Threshold from the inner years' predictions, F1 on the outer year."""
    _, _, threshold = evaluate_cached(FoldSubset(cache, inner), model)
    _, oof, _ = evaluate_cached(FoldSubset(cache, [outer]), model)
    return f1_score(oof["y"], oof["p"] >= threshold)


nested_rows = []
for outer in range(1, len(VAL_YEARS)):  # 2020 has no earlier year to tune on
    inner = list(range(outer))
    logistic_f1 = outer_f1(nb_cache, LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), inner, outer)

    for family, (cache, suggest, make, default_model) in nested_families.items():
        def objective(trial):
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(FoldSubset(pick_cache(cache, params), inner), make(params))
            return f1_score(oof["y"], oof["p"] >= threshold)

        # Tuned on the earlier years only; the outer year is never seen by the search
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=NESTED_TRIALS, show_progress_bar=False)
        best = study.best_params

        default_cache = cache if not callable(cache) else knn_caches["selected"]
        nested_rows.append({
            "outer_year": VAL_YEARS[outer],
            "family": family,
            "inner_years": len(inner),
            "nested_tuned": outer_f1(pick_cache(cache, best), make(best), inner, outer),
            "default": outer_f1(default_cache, default_model, inner, outer),
            "logistic": logistic_f1,
        })
        print(f"{VAL_YEARS[outer]} {family}: done")

nested = pd.DataFrame(nested_rows)
nested["gain_vs_default"] = nested["nested_tuned"] - nested["default"]
nested["gain_vs_logistic"] = nested["nested_tuned"] - nested["logistic"]

2021 hist_gb: done
2021 random_forest: done
2021 knn: done
2022 hist_gb: done
2022 random_forest: done
2022 knn: done
2023 hist_gb: done
2023 random_forest: done
2023 knn: done


In [94]:
display(nested.round(3))

,outer_year,family,inner_years,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
0,2021,hist_gb,1,0.769,0.725,0.764,0.045,0.005
1,2021,random_forest,1,0.755,0.742,0.764,0.013,-0.009
2,2021,knn,1,0.734,0.737,0.764,-0.003,-0.029
3,2022,hist_gb,2,0.702,0.689,0.714,0.013,-0.012
4,2022,random_forest,2,0.709,0.691,0.714,0.018,-0.005
5,2022,knn,2,0.708,0.682,0.714,0.026,-0.006
6,2023,hist_gb,3,0.769,0.737,0.752,0.031,0.016
7,2023,random_forest,3,0.770,0.741,0.752,0.029,0.018
8,2023,knn,3,0.751,0.762,0.752,-0.011,-0.001


In [95]:
display(nested.groupby("family")[["nested_tuned", "default", "logistic", "gain_vs_default", "gain_vs_logistic"]]
        .mean().round(3))

,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
family,,,,,
hist_gb,0.747,0.717,0.743,0.029,0.003
knn,0.731,0.727,0.743,0.004,-0.012
random_forest,0.745,0.725,0.743,0.020,0.001


When tuned only on earlier years, both ensembles still gain clearly over their defaults: +0.029 F1 on average for gradient boosting and +0.020 for the forest, positive in every outer year. They end level with logistic regression (+0.003 and +0.001 on average), ahead in 2023 (+0.016 and +0.018) and behind in 2022 (-0.012 and -0.005). KNN's tuning gain mostly disappears: +0.004 on average, negative in two of three years, and 0.012 below logistic regression. Its earlier gain came partly from choosing among many trials on the years it was scored on.

In [96]:
pd.set_option("display.width", 250)
cols = ["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]
print("--- 5.1 final board"); print(board_final[cols].round(3).to_string())
print("--- 5.1 by year"); print(fold_f1_final.round(3).to_string())
print("--- 5.2 rank correlation"); print(correlation.round(3).to_string())
print("--- 5.2 blends"); print(board_blend[cols[1:]].round(3).to_string())
print("--- 5.3 nested"); print(nested.round(3).to_string())
print(nested.groupby("family")[["nested_tuned", "default", "logistic", "gain_vs_default", "gain_vs_logistic"]].mean().round(3).to_string())

--- 5.1 final board
               n_features    f1  f1_diff  ci_low  ci_high  folds_worse  balanced_accuracy  roc_auc  log_loss  threshold              verdict
model                                                                                                                                       
hist_gb            38.000 0.753    0.003  -0.008    0.015            1              0.683    0.784     0.562      0.440  no clear difference
random_forest      38.000 0.751    0.001  -0.009    0.012            1              0.673    0.786     0.551      0.410  no clear difference
logistic           20.000 0.750    0.000     NaN      NaN            0              0.684    0.784     0.561      0.340            reference
knn                20.000 0.747   -0.003  -0.017    0.011            2              0.668    0.779     0.567      0.370  no clear difference
decision_tree      38.000 0.736   -0.015  -0.030   -0.001            4              0.657    0.756     0.642      0.400               

## 5.4. Choice

The rule set before the comparison keeps logistic regression unless an alternative passes the improvement test. None did: the three tuned finalists and the five blends all have intervals that include zero, and each is worse in at least one year. The nested check agrees: tuned gradient boosting and random forest match logistic regression when tuned on earlier years only, but do not beat it.

Logistic regression is therefore kept, with its threshold of 0.34. It uses 20 columns against 38 for the tree ensembles, and its coefficients can be read directly. The blend of all finalists is the runner-up: F1 0.755 against 0.750, with better probabilities (log-loss 0.546, ROC-AUC 0.791), but the difference is not clear.

In [97]:
# decision: keep logistic regression; no tuned model or blend passed the improvement rule
chosen_model = "logistic"
threshold = tuned_runs[chosen_model][2]
runner_up = "all finalists"

log.record(
    "D41", "default models",
    "no default model beats logistic regression",
    "Naive Bayes, LDA, KNN, decision tree and random forest at default settings: all clearly worse than "
    "logistic regression (F1 0.750) except untuned gradient boosting, whose interval included zero.",
    0, 0, carries={"default_f1": board_default["f1"].round(4).to_dict()},
)
log.record(
    "D42", "tuning",
    "tuned KNN, decision tree and random forest kept; Naive Bayes and LDA kept their defaults",
    "Each family was tuned with two 60-trial searches and compared with its own default under the decision rule. "
    "Calibration (Naive Bayes) and shrinkage (LDA) did not pass the rule.",
    0, 0, carries={"knn": knn_params, "decision_tree": tree_params, "random_forest": forest_params,
                   "tuned_f1": board_final["f1"].round(4).to_dict()},
)
log.record(
    "D43", "final model",
    f"retain {chosen_model}",
    f"Finalists {finalists} and five equal-weight blends all had F1 intervals including zero against logistic "
    f"regression. The nested check (tuned on earlier years only) gave gains of "
    f"{nested.groupby('family')['gain_vs_logistic'].mean().round(3).to_dict()} over logistic regression. "
    f"Runner-up: {runner_up} (F1 {board_blend.loc[runner_up, 'f1']:.3f}).",
    0, 0, carries={"chosen": chosen_model, "threshold": float(threshold), "runner_up": blend_members[runner_up]},
)
display(log.to_frame().tail(3))

,step,column,action,reason,rows_train,rows_test
38,D41,default models,no default model beats logistic regression,"Naive Bayes, LDA, KNN, decision tree and random forest at default settings: all clearly worse th...",0,0
39,D42,tuning,"tuned KNN, decision tree and random forest kept; Naive Bayes and LDA kept their defaults",Each family was tuned with two 60-trial searches and compared with its own default under the dec...,0,0
40,D43,final model,retain logistic,"Finalists ['hist_gb', 'random_forest', 'logistic', 'knn'] and five equal-weight blends all had F...",0,0


## Findings

| Candidate | F1 | Difference [95% interval] | ROC-AUC | Log-loss | Nested gain over logistic |
|---|---|---|---|---|---|
| Logistic regression | 0.750 | reference | 0.784 | 0.561 | — |
| Gradient boosting (tuned) | 0.753 | +0.003 [-0.008, 0.015] | 0.784 | 0.562 | +0.003 |
| Random forest (tuned) | 0.751 | +0.001 [-0.009, 0.012] | 0.786 | 0.551 | +0.001 |
| KNN (tuned) | 0.747 | -0.003 [-0.017, 0.011] | 0.779 | 0.567 | -0.012 |
| Blend of all finalists | 0.755 | +0.004 [-0.005, 0.013] | 0.791 | 0.546 | — |

- Four very different models reach the same F1 within 0.006. The limit looks like the information in the data, not the choice of algorithm.
- Blending improves the probabilities more than the labels: log-loss falls by up to 0.017 and ROC-AUC rises by up to 0.007, while F1 rises by at most 0.004.
- The ensembles' tuning gains survive the nested check; KNN's do not.
- Logistic regression is kept as the simplest model with no clear deficit. The blend of all finalists is recorded as the runner-up.